# proto_policy_search.ipynb

## 이 노트북을 만든 이유
이 노트북은 PolicyRec의 프로토타입 검색 흐름을 빠르게 검증하기 위해 만든 실험용 노트북입니다.
정규화된 정책 CSV( combined_normalized_v1_1_3.csv )를 불러와서, title, summary, category, region, target_group 등을 검색용 텍스트로 엮고, TF-IDF 기반으로 관련 정책 후보를 찾는 과정을 먼저 확인하는 목적입니다.

## 이 노트북에서 확인하는 것
- 정규화 CSV가 프로토타입 검색 입력으로 바로 사용 가능한지
- 사용자의 자연어 질문에 대해 관련 정책 Top-N이 어느 정도 맞게 나오는지
- Gemini API를 붙였을 때 검색 결과를 설명형 답변으로 연결할 수 있는지
- 추후 Streamlit 서비스 화면에 넣을 챗봇/검색 로직의 초안이 되는지

## 이 노트북의 위치
이 파일은 서비스 본 코드라기보다 검색 로직 검증용 프로토타입 노트북에 가깝습니다.
즉, PolicyRec_v1_1_3.ipynb가 데이터 정규화 기준 노트북이라면, proto_policy_search.ipynb는 그 결과물을 바탕으로 검색과 추천 흐름을 먼저 시험해보는 단계라고 보면 됩니다.

## 이후 연결 방향
- 데이터 기준: PolicyRec_v1_1_3.ipynb
- 프로토타입 웹: streamlit_app2.py
- 현재 노트북: 검색/챗봇 로직 검증용

정리하면, 이 노트북은 정규화된 정책 데이터를 실제로 어떻게 검색하고 추천에 연결할지 미리 시험해보는 중간 검증 노트북입니다.



In [ ]:
#!pip install scikit-learn google-genai pandas python-dotenv

In [ ]:
# ================================================
# 0. 라이브러리 불러오기
# ================================================

import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from google import genai

load_dotenv()  # .env 읽어서 환경변수로 등록

True

In [ ]:
# ================================================
# 1. CSV 불러오기
# ================================================

csv_path = Path("data/clean/combined_normalized_v1_1_3.csv")
df = pd.read_csv(csv_path, encoding="utf-8-sig")
print(f"{len(df)}건 로드됨")
df.head(2)

30건 로드됨


,source,source_id,title,summary,category,region,provider,target_group,target_age_min,target_age_max,start_date,end_date,detail_url
0,biz,PBLN_000000000121111,2026년 물산업 유망 해외프로젝트 발굴 지원사업 모집 공고,민관협력 해외진출 활성화를 위한 '2026년 유망 해외프로젝트 발굴 지원사업'을 다...,수출,전국,기후에너지환경부,중소기업,0,99,2026-04-16,2026-05-08,https://www.bizinfo.go.kr/sii/siia/selectSIIA2...
1,biz,PBLN_000000000121110,2026년 민관협력 해외사업 현지화 지원사업 모집 공고,"우수한 물기술과 역량을 보유한 중소기업과 공공부문이 민관협력체계를 구축하여, 물기업...",수출,전국,기후에너지환경부,중소기업,0,99,2026-04-16,2026-05-08,https://www.bizinfo.go.kr/sii/siia/selectSIIA2...


In [ ]:
# ================================================
# 2. 검색용 텍스트 만들기
# ================================================

def build_search_text(row):
    parts = [
        str(row.get("title", "")),
        str(row.get("summary", "")),
        str(row.get("category", "")),
        str(row.get("region", "")),
        str(row.get("target_group", "")),
    ]
    return " ".join(p for p in parts if p and p != "nan")

df["search_text"] = df.apply(build_search_text, axis=1)
df["search_text"].head(2)

0    2026년 물산업 유망 해외프로젝트 발굴 지원사업 모집 공고 민관협력 해외진출 활성...
1    2026년 민관협력 해외사업 현지화 지원사업 모집 공고 우수한 물기술과 역량을 보유...
Name: search_text, dtype: str

In [ ]:
# ================================================
# 3. TF-IDF 학습
# ================================================

vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
matrix = vectorizer.fit_transform(df["search_text"])
print("벡터 모양:", matrix.shape)

벡터 모양: (30, 7677)


In [ ]:
# ================================================
# 4. Top 5 검색 함수
# ================================================

def search_top5(query: str):
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, matrix)[0]
    top_idx = scores.argsort()[::-1][:5]
    return df.iloc[top_idx][["title", "provider", "region", "category"]].assign(score=scores[top_idx])

search_top5("청년 창업 지원")

,title,provider,region,category,score
10,2026년 5월 동네창업학교 교육생 모집 공고,충남신용보증재단,전국,창업교육,0.132407
24,제주 청년주권회의 운영,제주특별자치도 청년정책담당관,제주특별자치도,참여･기반,0.128849
25,제주 청년원탁회의 운영,제주특별자치도 청년정책담당관,제주특별자치도,참여･기반,0.124098
17,2026 지식재산 데이터 활용 창업 경진대회,한국특허정보원,전국,사업화,0.120596
18,창업On 희망On 아카데미 참여자 모집 공고,경북창조경제혁신센터,전국,멘토링ㆍ컨설팅ㆍ교육,0.110682


In [ ]:
# ================================================
# 5. Gemini 연결 (재시도 + fallback)
# ================================================

import time

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

def recommend(query: str, max_retries: int = 3):
    top = search_top5(query)
    candidates_text = "\n".join(
        f"- {row['title']} ({row['provider']}, {row['region']})"
        for _, row in top.iterrows()
    )
    prompt = (
        f"사용자 질문: {query}\n\n"
        f"후보 정책:\n{candidates_text}\n\n"
        f"각 정책이 왜 추천되는지 3문장으로 간단히 설명해줘."
    )

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-3-flash-preview",
                contents=prompt,
            )
            return top, response.text
        except Exception as e:
            print(f"[{attempt+1}/{max_retries}] 실패:", e)
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # 1초, 2초, 4초 점점 늘림

    # 그래도 실패하면 안정 모델(gemini-2.5-flash)로 한 번 더 시도
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
        )
        return top, response.text
    except Exception as e:
        print("최종 실패:", e)
        return top, None

top, explanation = recommend("청년 창업 지원")
print(explanation)
top

In [ ]:
# ================================================
# 6. 쿼리 임베딩 함수
# ================================================

# 확인할 것: output_dimensionality 값 (예: 768, 1536, 3072 중 무엇?)
EMBEDDING_DIM = 768  # 다른 팀원이 쓴 값으로 맞출 것

def embed_query(text: str):
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text,
        config={"output_dimensionality": EMBEDDING_DIM},
    )
    return response.embeddings[0].values

vec = embed_query("청년 창업 지원")
print("차원:", len(vec))
print("앞 5개:", vec[:5])

In [ ]:
# ================================================
# 7. 현재 30건을 로컬에서도 임베딩
# ================================================

import numpy as np

def embed_documents(texts: list[str]) -> np.ndarray:
    vectors = []
    for i, text in enumerate(texts):
        response = client.models.embed_content(
            model="gemini-embedding-001",
            contents=text,
            config={"output_dimensionality": EMBEDDING_DIM},
        )
        vectors.append(response.embeddings[0].values)
        if (i + 1) % 10 == 0:
            print(f"{i+1}/{len(texts)} 완료")
    return np.array(vectors)

doc_embeddings = embed_documents(df["search_text"].tolist())
print("문서 임베딩 모양:", doc_embeddings.shape)  # (30, 768) 기대

In [ ]:
# ================================================
# 8. 임베딩 기반 Top 5
# ================================================

def search_top5_embedding(query: str):
    query_vec = np.array(embed_query(query)).reshape(1, -1)
    scores = cosine_similarity(query_vec, doc_embeddings)[0]
    top_idx = scores.argsort()[::-1][:5]
    return df.iloc[top_idx][["title", "provider", "region", "category"]].assign(score=scores[top_idx])

search_top5_embedding("청년 창업 지원")

In [ ]:
# ================================================
# 9. TF-IDF vs 임베딩 비교
# ================================================

print("=== TF-IDF ===")
print(search_top5("청년 창업 지원")[["title", "score"]])
print("\n=== 임베딩 ===")
print(search_top5_embedding("청년 창업 지원")[["title", "score"]])